# One vectorised beam-search step with finished beams

Beam search keeps the $K$ highest-scoring partial sequences per example. Write one step of it as pure array code (it will sit inside a `scan`), including beams that have already finished.

Write `beam_step(scores, tokens, finished, log_probs, eos_id)`:

- `scores`: `(B, K)` float32, cumulative log-probabilities. May contain `-inf` (for example, at the first step every beam but one is set to `-inf` so the identical start beams are not selected $K$ times).
- `tokens`: `(B, K, t)` int, the tokens so far (`t >= 0`).
- `finished`: `(B, K)` bool.
- `log_probs`: `(B, K, V)` float32, next-token log-probabilities for each beam.

Rules:

1. Candidate $(k, v)$ has score `scores[b, k] + log_probs[b, k, v]` if beam $k$ is not finished. A finished beam has exactly one candidate, $v$ = `eos_id`, with its score **unchanged**; its other candidates score `-inf`.
2. Flatten the candidates to index $k \cdot V + v$ and keep the $K$ best by score, descending. Equal scores are ordered by **lower flat index first**.
3. For each kept candidate: `parent = index // V`, `token = index % V`; the new history is `tokens[b, parent]` with `token` appended; it is finished if its parent was finished or `token == eos_id`.

Return `(new_scores, new_tokens, new_finished, parent)` with shapes `(B, K)`, `(B, K, t + 1)`, `(B, K)`, `(B, K)`. Scores float32, tokens and parents int32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def beam_step(scores, tokens, finished, log_probs, eos_id):
    """Return (new_scores (B, K), new_tokens (B, K, t+1), new_finished (B, K), parent (B, K))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, {"__f__": "-inf"}]], "dtype": "float64"}, {"__nd__": [[[], []]], "dtype": "int64"}, {"__nd__": [[false, false]], "dtype": "bool"}, {"__nd__": [[[-0.6931471805599453, -1.2039728043259361, -1.6094379124341003], [-0.916290731874155, -0.916290731874155, -1.6094379124341003]]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[-0.6931471824645996, -1.2039728164672852]], "dtype": "float32"}, {"__nd__": [[[0], [1]]], "dtype": "int32"}, {"__nd__": [[false, false]], "dtype": "bool"}, {"__nd__": [[0, 0]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-1.0, -2.0]], "dtype": "float64"}, {"__nd__": [[[0], [1]]], "dtype": "int64"}, {"__nd__": [[true, false]], "dtype": "bool"}, {"__nd__": [[[-2.3025850929940455, -2.3025850929940455, -0.2231435513142097], [-0.5108256237659907, -1.2039728043259361, -2.3025850929940455]]], "dtype": "float64"}, 2], "e": {"__tup__": [{"__nd__": [[-1.0, -2.5108256340026855]], "dtype": "float32"}, {"__nd__": [[[0, 2], [1, 0]]], "dtype": "int32"}, {"__nd__": [[true, false]], "dtype": "bool"}, {"__nd__": [[0, 1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[0.0, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float64"}, {"__nd__": [[[], [], []]], "dtype": "int64"}, {"__nd__": [[false, false, false]], "dtype": "bool"}, {"__nd__": [[[-1.3862943611198906, -1.3862943611198906, -1.3862943611198906, -1.3862943611198906], [-1.3862943611198906, -1.3862943611198906, -1.3862943611198906, -1.3862943611198906], [-1.3862943611198906, -1.3862943611198906, -1.3862943611198906, -1.3862943611198906]]], "dtype": "float64"}, 0], "e": {"__tup__": [{"__nd__": [[-1.3862943649291992, -1.3862943649291992, -1.3862943649291992]], "dtype": "float32"}, {"__nd__": [[[0], [1], [2]]], "dtype": "int32"}, {"__nd__": [[true, false, false]], "dtype": "bool"}, {"__nd__": [[0, 0, 0]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-1.83106492443329, -0.5582151522076139, -0.975150596580685], [-1.130303657406008, -0.3759831707143508, -0.028160785573402004]], "dtype": "float64"}, {"__nd__": [[[3, 4], [3, 3], [2, 4]], [[4, 2], [3, 1], [4, 2]]], "dtype": "int64"}, {"__nd__": [[false, false, false], [false, false, false]], "dtype": "bool"}, {"__nd__": [[[-0.7741323838297132, -1.7934015222884354, -2.5340346384859425, -1.6336912503739132, -2.3232250267440553], [-1.6652183363722426, -2.5143072753803364, -3.0548947635894947, -0.5649267633066897, -2.1680345829999426], [-1.4682481970150887, -2.1873417265616744, -2.1314097995924217, -1.940238558199158, -0.9285728980576623]], [[-1.9750660145003727, -2.6509279035979225, -3.747493510197317, -0.3973153591092906, -2.3542641188533415], [-0.8393184864798665, -2.329841686169918, -2.0259420064563156, -1.1735259951674069, -3.5220123892992783], [-3.489180702822404, -2.315025064290843, -0.2929860815589298, -2.2906978885776357, -3.7515176275928264]]], "dtype": "float64"}, 4], "e": {"__tup__": [{"__nd__": [[-1.123141884803772, -1.9037234783172607, -2.223433494567871], [-0.3211468756198883, -1.215301752090454, -1.5276190042495728]], "dtype": "float32"}, {"__nd__": [[[3, 3, 3], [2, 4, 4], [3, 3, 0]], [[4, 2, 2], [3, 1, 0], [4, 2, 3]]], "dtype": "int32"}, {"__nd__": [[false, true, false], [false, false, false]], "dtype": "bool"}, {"__nd__": [[1, 2, 1], [2, 1, 0]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-0.14474158718048616, -2.1272421486426176, -0.07160353326786321, -1.3014307034193155], [-0.4752438706252675, -1.3628007616423388, -0.6559437836687712, -0.06107614316095217], [-1.0811606334038053, -0.00906485772765128, -0.6978304985347987, -0.0721955736091649]], "dtype": "float64"}, {"__nd__": [[[2, 2, 3], [2, 5, 4], [0, 1, 2], [4, 3, 1]], [[4, 2, 1], [4, 2, 2], [0, 4, 3], [1, 4, 1]], [[4, 0, 2], [5, 5, 4], [3, 4, 1], [5, 0, 0]]], "dtype": "int64"}, {"__nd__": [[false, true, true, true], [true, true, false, true], [false, false, true, false]], "dtype": "bool"}, {"__nd__": [[[-3.57019039479524, -4.599166570617298, -6.156467268797375, -4.25349962480501, -0.0729023646988245, -4.150054121712097], [-1.0348956022083415, -4.0975654336745375, -1.0248490006784055, -4.205487184155349, -2.364056063630049, -1.8305755402737909], [-0.49118513433979727, -3.5980000808636694, -2.3912083931435184, -3.4396655846050415, -1.9602372414399087, -2.34027289561953], [-2.168320425444426, -0.43684833301932136, -3.748517062831703, -3.25286683822425, -2.82300712515407, -2.1377639741257495]], [[-5.510542011802849, -2.881541123013964, -6.070644046083469, -0.35853527810467645, -3.7890624691492216, -1.5311707408642223], [-6.237811390369706, -4.698063339505695, -3.7995045384875312, -0.516051507650618, -3.8245901608095334, -1.0559808291592065], [-4.528501155873316, -0.09620061845338146, -4.459684947596683, -6.659834926879924, -7.05329391763785, -2.6999413778233583], [-3.9110126653707473, -2.9854997135541756, -3.624468850209262, -2.5029528112697106, -0.22266340729607625, -3.883674129772224]], [[-2.599882644864866, -3.933201599346496, -6.610505031025195, -3.704064646665041, -0.13732155612940522, -4.770333910756668], [-0.1358053443105831, -3.6096326384963042, -3.807560312866663, -2.8377305436178055, -4.6795782024416255, -4.617036156680837], [-3.483064907010092, -7.590663902071716, -6.433739604343363, -8.514538959556058, -5.919196084916044, -0.036365898869760827], [-0.5346056428257842, -4.788075042936942, -6.450657314025274, -1.7925580560204484, -1.5070746926285254, -4.128875871063846]]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[-0.07160353660583496, -0.21764394640922546, -1.3014307022094727, -2.127242088317871], [-0.06107614189386368, -0.47524386644363403, -0.752144455909729, -1.3628007173538208], [-0.14487019181251526, -0.6068012118339539, -0.6978304982185364, -1.218482255935669]], "dtype": "float32"}, {"__nd__": [[[0, 1, 2, 1], [2, 2, 3, 4], [4, 3, 1, 1], [2, 5, 4, 1]], [[1, 4, 1, 1], [4, 2, 1, 1], [0, 4, 3, 1], [4, 2, 2, 1]], [[5, 5, 4, 0], [5, 0, 0, 0], [3, 4, 1, 1], [4, 0, 2, 4]]], "dtype": "int32"}, {"__nd__": [[true, false, true, true], [true, true, true, true], [false, false, true, false]], "dtype": "bool"}, {"__nd__": [[2, 0, 3, 1], [3, 0, 2, 1], [1, 3, 2, 0]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-0.5, -0.7]], "dtype": "float64"}, {"__nd__": [[[3, 1], [2, 1]]], "dtype": "int64"}, {"__nd__": [[true, true]], "dtype": "bool"}, {"__nd__": [[[-0.9209106681364871, -1.8341647976837763, -1.5355419540095963, -1.4838723965052636], [-0.9865579324635783, -2.531092247637667, -2.785512489216025, -0.7218088475909954]]], "dtype": "float64"}, 1], "e": {"__tup__": [{"__nd__": [[-0.5, -0.699999988079071]], "dtype": "float32"}, {"__nd__": [[[3, 1, 1], [2, 1, 1]]], "dtype": "int32"}, {"__nd__": [[true, true]], "dtype": "bool"}, {"__nd__": [[0, 1]], "dtype": "int32"}]}}, {"a": [{"__nd__": [[-0.679163185354929, -2.3730313222847865], [-0.2100293425539758, -0.2543160276719131]], "dtype": "float64"}, {"__nd__": [[[0, 1, 2, 1], [0, 1, 1, 2]], [[2, 0, 0, 1], [0, 1, 2, 2]]], "dtype": "int64"}, {"__nd__": [[false, true], [true, false]], "dtype": "bool"}, {"__nd__": [[[-1.1597834173963848, -0.7938644886365706, -1.4509326661231972], [-2.16555477936716, -0.9247075146439391, -0.7160754499268103]], [[-1.3704129761282047, -0.3941800066796457, -2.6343544859137626], [-0.7627345795772076, -1.2050721693573057, -1.4526892160573819]]], "dtype": "float64"}, 0], "e": {"__tup__": [{"__nd__": [[-1.4730277061462402, -1.8389465808868408], [-0.21002934873104095, -1.0170506238937378]], "dtype": "float32"}, {"__nd__": [[[0, 1, 2, 1, 1], [0, 1, 2, 1, 0]], [[2, 0, 0, 1, 0], [0, 1, 2, 2, 0]]], "dtype": "int32"}, {"__nd__": [[false, true], [true, true]], "dtype": "bool"}, {"__nd__": [[0, 0], [0, 1]], "dtype": "int32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(beam_step, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")